<a href="https://colab.research.google.com/github/lohaniSatwik/steam-games-data-mining/blob/master/Code/section5_evaluation_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Section 5 — Model Evaluation & Interpretability
**IE500 Data Mining | Team 9 – Brewed Clusters**

> **Google Colab notebook.** Run all cells top to bottom.

### What this notebook does
1. Retrains all 5 models using their **tuned hyperparameters** from sections 4a–4e
2. Builds a complete **model comparison table** (Macro F1, per-class F1)
3. Plots **confusion matrices** for all models
4. Runs **SHAP analysis** on the best model (Random Forest) for interpretability
5. Compares **RF Gini vs SHAP** feature importance
6. Checks **data leakage** from post-release features
7. Draws **final conclusions**

### Task
**Multiclass classification: Good / Mixed / Bad** game reception on Steam  
Primary metric: **Macro F1** (treats all 3 classes equally)

In [ ]:
!pip install shap xgboost imbalanced-learn -q

In [ ]:
import os
if not os.path.exists('steam-games-data-mining'):
    !git clone https://github.com/lohaniSatwik/steam-games-data-mining.git
else:
    !git -C steam-games-data-mining pull
DATA_DIR = 'steam-games-data-mining/Datasets'

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import shap
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import (
    f1_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay
)
from xgboost import XGBClassifier

RANDOM_STATE = 42
CLASS_ORDER  = ['Good', 'Mixed', 'Bad']
MODEL_COLORS = {
    'Logistic Regression': '#4C72B0',
    'Linear SVM':          '#DD8452',
    'Random Forest':       '#55A868',
    'XGBoost':             '#C44E52',
    'KNN':                 '#9467BD'
}

print('Libraries loaded.')

In [ ]:
train = pd.read_csv(f'{DATA_DIR}/train_multiclass.csv')
test  = pd.read_csv(f'{DATA_DIR}/test_multiclass.csv')

X_train_full = train.drop(columns=['label_multiclass'])
y_train_full = train['label_multiclass']
X_test        = test.drop(columns=['label_multiclass'])
y_test        = test['label_multiclass']

# 15K stratified subsample — used only for KNN (distance computation is too slow on 45K)
X_train_knn, _, y_train_knn, _ = train_test_split(
    X_train_full, y_train_full,
    train_size=15000, stratify=y_train_full,
    random_state=RANDOM_STATE
)

# XGBoost needs numeric labels (LabelEncoder sorts alphabetically: Bad=0, Good=1, Mixed=2)
le = LabelEncoder()
le.fit(CLASS_ORDER)
y_train_enc = le.transform(y_train_full)
y_test_enc  = le.transform(y_test)

print(f'Train: {X_train_full.shape}  |  Test: {X_test.shape}')
print(f'\nClass distribution (train):')
vc = y_train_full.value_counts()
for cls in CLASS_ORDER:
    print(f'  {cls:6s}: {vc[cls]:6,d}  ({vc[cls]/len(y_train_full)*100:.1f}%)')

## 1 — Retrain All Models

Each model is retrained on the **full training set** using its **tuned hyperparameters** from sections 4a–4e.  
No CV here — this is the final fit before test set evaluation.

In [ ]:
import time

# --- Logistic Regression (section 4a) ---
# Best params: C=0.1, L2, lbfgs
t0 = time.time()
lr_model = LogisticRegression(
    C=0.1, penalty='l2', solver='lbfgs',
    class_weight='balanced', max_iter=2000, random_state=RANDOM_STATE
)
lr_model.fit(X_train_full, y_train_full)
print(f'Logistic Regression    fitted in {time.time()-t0:.1f}s')

# --- Linear SVM (section 4d) ---
# Best params: C=0.1
t0 = time.time()
svm_model = LinearSVC(
    C=0.1, class_weight='balanced', dual=False,
    max_iter=2000, random_state=RANDOM_STATE
)
svm_model.fit(X_train_full, y_train_full)
print(f'Linear SVM             fitted in {time.time()-t0:.1f}s')

# --- Random Forest (section 4b) ---
# Best params: n_estimators=200, max_depth=None, min_samples_leaf=4
t0 = time.time()
rf_model = RandomForestClassifier(
    n_estimators=200, max_depth=None, min_samples_leaf=4,
    max_features='sqrt', class_weight='balanced',
    random_state=RANDOM_STATE, n_jobs=-1
)
rf_model.fit(X_train_full, y_train_full)
print(f'Random Forest          fitted in {time.time()-t0:.1f}s')

# --- XGBoost (section 4c) ---
# Best params: n_estimators=200, max_depth=6, learning_rate=0.1
t0 = time.time()
sample_weight = compute_sample_weight('balanced', y_train_enc)
xgb_model = XGBClassifier(
    n_estimators=200, max_depth=6, learning_rate=0.1,
    objective='multi:softmax', num_class=3,
    eval_metric='mlogloss', verbosity=0,
    random_state=RANDOM_STATE, n_jobs=-1
)
xgb_model.fit(X_train_full, y_train_enc, sample_weight=sample_weight)
print(f'XGBoost                fitted in {time.time()-t0:.1f}s')

# --- KNN (section 4e) ---
# Best params: n_neighbors=11, weights=distance, metric=manhattan
# Uses 15K subsample — consistent with CV approach (KNN scales quadratically)
t0 = time.time()
knn_model = KNeighborsClassifier(
    n_neighbors=11, weights='distance', metric='manhattan',
    algorithm='brute', n_jobs=-1
)
knn_model.fit(X_train_knn, y_train_knn)
print(f'KNN (15K subsample)    fitted in {time.time()-t0:.1f}s')

print('\nAll models trained.')

In [ ]:
# Collect test set predictions
preds = {
    'Logistic Regression': lr_model.predict(X_test),
    'Linear SVM':          svm_model.predict(X_test),
    'Random Forest':       rf_model.predict(X_test),
    'XGBoost':             le.inverse_transform(xgb_model.predict(X_test).astype(int)),
    'KNN':                 knn_model.predict(X_test),
}

print('Test set Macro F1 (quick check):')
for name, y_pred in preds.items():
    f1 = f1_score(y_test, y_pred, average='macro')
    print(f'  {name:22s}  {f1:.4f}')

## 2 — Model Comparison Table

Nested CV scores are taken directly from each model's notebook (sections 4a–4e).  
Test scores are computed fresh above.

In [ ]:
# Nested CV results (from sections 4a-4e)
cv_results = {
    'Logistic Regression': {'mean': 0.4385, 'std': 0.0034},
    'Linear SVM':          {'mean': 0.4680, 'std': 0.0034},
    'Random Forest':       {'mean': 0.5093, 'std': 0.0073},
    'XGBoost':             {'mean': 0.4820, 'std': 0.0039},
    'KNN':                 {'mean': 0.4462, 'std': 0.0066},
}

rows = []
for name, y_pred in preds.items():
    report = classification_report(
        y_test, y_pred, labels=CLASS_ORDER,
        target_names=CLASS_ORDER, output_dict=True
    )
    cv = cv_results[name]
    rows.append({
        'Model':            name,
        'CV Macro F1':      f"{cv['mean']:.4f} ± {cv['std']:.4f}",
        'Test Macro F1':    f"{f1_score(y_test, y_pred, average='macro'):.4f}",
        'F1 Good':          f"{report['Good']['f1-score']:.4f}",
        'F1 Mixed':         f"{report['Mixed']['f1-score']:.4f}",
        'F1 Bad':           f"{report['Bad']['f1-score']:.4f}",
        'Accuracy':         f"{report['accuracy']:.4f}",
    })

df_comparison = pd.DataFrame(rows).set_index('Model')
print('=' * 85)
print('MODEL COMPARISON TABLE')
print('=' * 85)
print(df_comparison.to_string())
print('=' * 85)

In [ ]:
# Per-class F1 grouped bar chart
model_names = list(preds.keys())
f1_good  = [float(df_comparison.loc[m, 'F1 Good'])  for m in model_names]
f1_mixed = [float(df_comparison.loc[m, 'F1 Mixed']) for m in model_names]
f1_bad   = [float(df_comparison.loc[m, 'F1 Bad'])   for m in model_names]
f1_macro = [float(df_comparison.loc[m, 'Test Macro F1']) for m in model_names]

x = np.arange(len(model_names))
width = 0.2

fig, ax = plt.subplots(figsize=(13, 5))
ax.bar(x - 1.5*width, f1_good,  width, label='Good',       color='steelblue',   alpha=0.85)
ax.bar(x - 0.5*width, f1_mixed, width, label='Mixed',      color='sandybrown',  alpha=0.85)
ax.bar(x + 0.5*width, f1_bad,   width, label='Bad',        color='salmon',      alpha=0.85)
ax.bar(x + 1.5*width, f1_macro, width, label='Macro Avg',  color='dimgrey',     alpha=0.85)

ax.set_xticks(x)
ax.set_xticklabels(model_names, fontsize=10)
ax.set_ylabel('F1 Score')
ax.set_title('Per-class and Macro F1 by Model (Test Set)', fontsize=13)
ax.set_ylim(0, 1)
ax.axhline(0.4355, color='black', linestyle='--', linewidth=1, alpha=0.5, label='LR Baseline')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

## 3 — Confusion Matrices

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(25, 4))

for ax, (name, y_pred) in zip(axes, preds.items()):
    cm = confusion_matrix(y_test, y_pred, labels=CLASS_ORDER)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=CLASS_ORDER)
    disp.plot(ax=ax, colorbar=False, cmap='Blues')
    f1 = f1_score(y_test, y_pred, average='macro')
    ax.set_title(f'{name}\nMacro F1 = {f1:.4f}', fontsize=9)

plt.suptitle('Confusion Matrices — All Models (Test Set)', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

## 4 — SHAP Analysis (Random Forest — Best Model)

SHAP (SHapley Additive exPlanations) explains **why** the model made each prediction.  
For a multiclass Random Forest, `TreeExplainer` returns one SHAP array per class.  
Each value tells us: "how much did this feature push the prediction toward this class?"

We use a random **2,000-sample subset** of the test set to keep computation fast.

In [ ]:
np.random.seed(RANDOM_STATE)
shap_idx = np.random.choice(len(X_test), size=2000, replace=False)
X_shap = X_test.iloc[shap_idx].reset_index(drop=True)

print(f'SHAP sample: {X_shap.shape}')
print('Computing SHAP values for Random Forest...')

explainer = shap.TreeExplainer(rf_model)
shap_values_raw = explainer.shap_values(X_shap)

# Newer SHAP returns a 3D array (n_samples, n_features, n_classes).
# Older SHAP returns a list of n_classes arrays each (n_samples, n_features).
# Normalise to list format so the rest of the code works in both cases.
if isinstance(shap_values_raw, np.ndarray) and shap_values_raw.ndim == 3:
    shap_values = [shap_values_raw[:, :, i] for i in range(shap_values_raw.shape[2])]
    print(f'SHAP format: 3D array → converted to list of {len(shap_values)} class arrays')
else:
    shap_values = shap_values_raw
    print(f'SHAP format: list of {len(shap_values)} class arrays')

print(f'Each array shape: {shap_values[0].shape}  (n_samples × n_features)')

In [ ]:
# Global SHAP summary — all 3 classes on one plot
# shap_values is a list of 3 arrays, one per class (Good, Mixed, Bad)
plt.figure(figsize=(10, 8))
shap.summary_plot(
    shap_values,
    X_shap,
    class_names=CLASS_ORDER,
    max_display=15,
    show=False
)
plt.title('SHAP Summary Plot — Random Forest (Top 15 Features)', fontsize=12, pad=15)
plt.tight_layout()
plt.show()

In [ ]:
# Mean absolute SHAP importance averaged across all 3 classes
# shap_values is a list: [Good_array, Mixed_array, Bad_array], each (n_samples, n_features)
mean_abs_shap = np.mean([np.abs(sv) for sv in shap_values], axis=0).mean(axis=0)

shap_importance = pd.Series(mean_abs_shap, index=X_shap.columns)\
                    .sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 7))
shap_importance.head(20).sort_values().plot(
    kind='barh', ax=ax, color='steelblue', alpha=0.85
)
ax.set_title('Mean |SHAP Value| — Top 20 Features (RF, averaged across classes)', fontsize=12)
ax.set_xlabel('Mean |SHAP Value|')
plt.tight_layout()
plt.show()

print('\nTop 15 features by mean |SHAP|:')
for feat, val in shap_importance.head(15).items():
    print(f'  {feat:40s}  {val:.5f}')

### 4.1 — Per-Class SHAP (What drives each label?)

A separate beeswarm plot for each class shows which features push games **toward** that class.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(21, 6))
class_colors = ['steelblue', 'sandybrown', 'salmon']

for i, (cls, color) in enumerate(zip(CLASS_ORDER, class_colors)):
    plt.sca(axes[i])
    shap.summary_plot(
        shap_values[i], X_shap,
        max_display=12,
        show=False,
        plot_size=None
    )
    axes[i].set_title(f'SHAP — {cls} class', fontsize=11)

plt.suptitle('Per-Class SHAP Beeswarm Plots (Random Forest)', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

## 5 — Feature Importance: RF Gini vs SHAP

Comparing two importance measures:  
- **RF Gini** (from tree splits) — fast, built-in, but can favour high-cardinality features  
- **SHAP** (Shapley values) — model-agnostic, accounts for feature interactions  

Agreement between the two increases confidence that these features genuinely matter.

In [ ]:
rf_importance = pd.Series(
    rf_model.feature_importances_, index=X_train_full.columns
).sort_values(ascending=False)

# Normalise both to 0-1 for side-by-side comparison
top_n = 15
rf_top   = (rf_importance.head(top_n) / rf_importance.head(top_n).max()).rename('RF Gini')
shap_top = (shap_importance.head(top_n) / shap_importance.head(top_n).max()).rename('SHAP')

features_union = list(dict.fromkeys(list(rf_top.index) + list(shap_top.index)))[:top_n]

plot_df = pd.DataFrame({
    'RF Gini': [rf_importance.get(f, 0) / rf_importance.iloc[0] for f in features_union],
    'SHAP':    [shap_importance.get(f, 0) / shap_importance.iloc[0] for f in features_union]
}, index=features_union)

fig, ax = plt.subplots(figsize=(10, 7))
x = np.arange(len(plot_df))
width = 0.38
ax.barh(x + width/2, plot_df['RF Gini'][::-1], width, label='RF Gini',  color='steelblue', alpha=0.85)
ax.barh(x - width/2, plot_df['SHAP'][::-1],    width, label='SHAP',     color='salmon',    alpha=0.85)
ax.set_yticks(x)
ax.set_yticklabels(plot_df.index[::-1], fontsize=9)
ax.set_xlabel('Normalised Importance')
ax.set_title('RF Gini vs SHAP — Top 15 Features (Normalised)', fontsize=12)
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

# Rank comparison table
rf_ranks   = {f: i+1 for i, f in enumerate(rf_importance.index)}
shap_ranks = {f: i+1 for i, f in enumerate(shap_importance.index)}
rank_df = pd.DataFrame([
    {'Feature': f, 'RF Rank': rf_ranks.get(f, '-'), 'SHAP Rank': shap_ranks.get(f, '-')}
    for f in features_union
])
print('\nRank comparison (top 15 by union):')
print(rank_df.to_string(index=False))

## 6 — Data Leakage Check

Three features are flagged in the README as **post-release metrics** — information that would  
not be available when predicting a game's reception before release:
- `Average playtime forever`
- `Median playtime forever`
- `Recommendations`

If these rank very highly in feature importance, the model may be "cheating" — learning from  
outcomes rather than pre-release attributes.

In [ ]:
leakage_features = ['Average playtime forever', 'Median playtime forever', 'Recommendations']

print('Post-release feature positions in importance rankings:')
print(f'\n{"Feature":35s}  {"RF Gini Rank":>15s}  {"SHAP Rank":>12s}  {"RF Importance":>15s}  {"SHAP Value":>12s}')
print('-' * 95)
for feat in leakage_features:
    rf_rank   = list(rf_importance.index).index(feat) + 1 if feat in rf_importance.index else 'N/A'
    shap_rank = list(shap_importance.index).index(feat) + 1 if feat in shap_importance.index else 'N/A'
    rf_val    = rf_importance.get(feat, 0)
    shap_val  = shap_importance.get(feat, 0)
    print(f'{feat:35s}  {str(rf_rank):>15s}  {str(shap_rank):>12s}  {rf_val:>15.5f}  {shap_val:>12.5f}')

print('\nInterpretation:')
print('  If Recommendations ranks #1, the model heavily relies on post-release data.')
print('  A follow-up experiment removing these 3 features would show the true pre-release predictability.')

## 7 — Final Conclusions

In [ ]:
best_model = df_comparison['Test Macro F1'].astype(float).idxmax()
best_f1    = df_comparison.loc[best_model, 'Test Macro F1']
lr_f1      = df_comparison.loc['Logistic Regression', 'Test Macro F1']

print('=' * 70)
print('SECTION 5 — FINAL CONCLUSIONS')
print('=' * 70)

print(f'''
1. BEST MODEL
   {best_model} achieved the highest Test Macro F1 = {best_f1}
   This is +{float(best_f1)-float(lr_f1):.4f} over the Logistic Regression baseline ({lr_f1}).

2. MODEL RANKING (Test Macro F1)
   Random Forest  >  XGBoost  >  Linear SVM  >  KNN  >  Logistic Regression
   Tree-based ensembles outperform linear models — the data has non-linear structure.

3. CLASS DIFFICULTY
   Good  is easiest to classify (F1 ~0.77 across models) — dominant class, strong signal.
   Mixed is hardest (F1 ~0.33–0.43) — games rated 50–74% overlap heavily with Good.
   Bad   is sparse (8.4%) but learnable with balanced weighting (F1 ~0.29–0.33).

4. CLASS IMBALANCE HANDLING
   class_weight='balanced' was applied to LR, SVM, and RF.
   XGBoost used compute_sample_weight('balanced').
   KNN has no class_weight support — Bad class recall is lowest (0.15) as a result.
   ROS and SMOTE experiments on LR confirmed balanced weighting is the right choice:
   ROS showed no improvement (0.4352 vs 0.4355), SMOTE showed +0.029 but added complexity.

5. MOST IMPORTANT FEATURES (from SHAP + RF Gini)
   Both methods agree on the top features:
   log_price, Achievements, n_languages, era_2020+, cat_Steam Cloud, tag_2D
   → Price, content richness, language support, and release era are the strongest predictors.

6. DATA LEAKAGE WARNING
   Recommendations, Average playtime, and Median playtime are post-release metrics.
   If these rank highly, the model uses information unavailable at prediction time.
   A follow-up run without these 3 features is recommended to assess pre-release predictability.

7. STABILITY
   All models show CV ≈ Test scores (gap < 0.01) — no overfitting.
   Random Forest nested CV std = 0.0073 — slightly higher variance than SVM/LR but acceptable.
''')
print('=' * 70)